In [18]:
import numpy as np
import pandas as pd
import csv
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import requests
from time import sleep
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import librosa

In [19]:
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id="c86c1e75263b40c7a1e413688ba800ac",
    client_secret="2e8781f731574ad49b9babe39e1de536",
    redirect_uri="http://127.0.0.1:8888/callback",
    scope="playlist-read-private playlist-read-collaborative"
))
user = sp.current_user()
print(user['id'])
results = sp.user_playlist_tracks(user['id'], "spotify:playlist:7qErWug3PkyUZH18Dqro80")
results2 = sp.user_playlist_tracks(user['id'], "spotify:playlist:5NZz6Y1Gvvr4faFuwMyemv")

tracks_data = []
while results:
    for item in results['items']:
        track = item['item']
        
        if track is None:
            continue
        
        tracks_data.append({
            'track_id': track['id'],
            'track_name': track['name'],
            'artist': track['artists'][0]['name'],
            'label': '1'
        })
    results = sp.next(results) if results['next'] else None
for item in results2['items']:
    track = item['item']
    
    if track is None:
        continue
    
    tracks_data.append({
        'track_id': track['id'],
        'track_name': track['name'],
        'artist': track['artists'][0]['name'],
        'label': '0'
    })

31vh2dchvotstorjhuxcxqvsosyi


In [20]:
with open('tracks.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['track_id', 'track_name', 'artist', 'label'])
    writer.writeheader()
    writer.writerows(tracks_data)

print(f"Exported {len(tracks_data)} tracks")

Exported 650 tracks


In [21]:
df = pd.read_csv('tracks.csv', encoding='latin-1')

positives = df[df['label'] == 1].sample(n=55, random_state=42)
negatives = df[df['label'] == 0]

balanced = pd.concat([positives, negatives]).sample(frac=1, random_state=42).reset_index(drop=True)
balanced.to_csv('tracks_balanced.csv', index=False)

print(balanced['label'].value_counts())
print(balanced.head())

label
0    55
1    55
Name: count, dtype: int64
                 track_id                    track_name         artist  label
0  5hRzAbY2AAO258hL6oqsqO                       Love Me       The 1975      0
1  08ikZNbHrmw7gxgB4vGpiv                   Joshua Tree  Cautious Clay      1
2  4uNAhebHwj78KP8FpBEt3i       When The World Was Mine       Bad Suns      1
3  0NEWcnBD0mb41XNA0jQQmB  I Hope It's Cold In New York     The Wrecks      0
4  2aNM3HvgJvGXIKBdBmWe1P                      Levitate   Atlas Genius      0


In [22]:
LASTFM_API_KEY = "2f78fb65cc6c8c71f822f5b06efef293"

def get_lastfm_tags(artist):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.gettoptags",
        "artist": artist,
        "api_key": LASTFM_API_KEY,
        "format": "json"
    }
    r = requests.get(url, params=params)
    data = r.json()
    try:
        tags = [t['name'].lower() for t in data['toptags']['tag'][:5]]
        return tags
    except:
        return []

balanced['tags'] = balanced['artist'].apply(lambda x: get_lastfm_tags(x))
sleep(0.25)  # rate limit

print(balanced[['artist', 'tags']].head(10))

                 artist                                               tags
0              The 1975    [indie, indie rock, british, alternative, rock]
1         Cautious Clay  [rnb, soul, alternative rnb, pop, singer-songw...
2              Bad Suns   [indie rock, indie, rock, indie pop, space rock]
3            The Wrecks   [indie rock, rock, my top songs, punk, pop rock]
4          Atlas Genius  [indie rock, electronic, indie, synthpop, aust...
5  Two Door Cinema Club  [indie, electronic, british, alternative, synt...
6         WALK THE MOON  [indie rock, indie, indie pop, american, alter...
7             Grayscale   [pop punk, rock, gothic metal, emo, alternative]
8         Knox Hamilton   [indie rock, usa, powerpop, christian, pop rock]
9              Bad Suns   [indie rock, indie, rock, indie pop, space rock]


In [23]:
mlb = MultiLabelBinarizer()
tag_features = mlb.fit_transform(balanced['tags'])
tag_df = pd.DataFrame(tag_features, columns=mlb.classes_)

print(tag_df.shape)
print(mlb.classes_)
X = tag_df
y = balanced['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

(110, 74)
['60s' '80s' 'alt pop' 'alternative' 'alternative rnb' 'alternative rock'
 'ambient' 'american' 'americana' 'australian' 'british' 'britpop'
 'canadian' 'christian' 'contemporary jazz' 'dance' 'dream pop' 'electro'
 'electronic' 'electronica' 'electropop' 'emo' 'england' 'experimental'
 'favs' 'female vocalists' 'folk' 'funk' 'good' 'gothic metal'
 'guitar pop' 'hip-hop' 'house' 'indie' 'indie folk' 'indie pop'
 'indie rock' 'indietronica' 'jazz' 'lastfmsc' 'los angeles' 'manchester'
 'my top songs' 'new orleans' 'new wave' 'oi' 'pop' 'pop punk' 'pop rock'
 'post-punk' 'powerpop' 'progressive metal' 'psychedelic'
 'psychedelic rock' 'punk' 'punk rock' 'rnb' 'rock' 'scratch'
 'singer-songwriter' 'skinhead' 'smooth jazz' 'soul' 'space rock'
 'street punk' 'swedish' 'synth indie rock' 'synth pop' 'synthpop'
 'trip-hop' 'turntablism' 'united states' 'upbeat' 'usa']
              precision    recall  f1-score   support

           0       0.67      0.60      0.63        10
       

In [ ]:
def extract_features(filepath):
    y, sr = librosa.load(filepath, duration=30)
    return {
        'bpm': librosa.beat.beat_track(y=y, sr=sr)[0],
        'spectral_centroid': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
        'chroma': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
        'energy': np.mean(librosa.feature.rms(y=y)),
        'zcr': np.mean(librosa.feature.zero_crossing_rate(y=y))
    }

print(extract_features(r"C:\Users\JoshQ\test.mp3"))

{'bpm': array([129.19921875]), 'spectral_centroid': np.float64(2266.909751649446), 'chroma': np.float32(0.33997396), 'energy': np.float32(0.22182012), 'zcr': np.float64(0.09747031008126934)}


: 